<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_5_Decision_Boundaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Part 5
## Decision Boundaries and Feature Importance

**Author:** Brad Sheese

---

## Introduction: How Models "See" the Data

We have reached the final notebook in our classification series! We have learned how to train models, evaluate them using complex metrics, and pick the best algorithm via cross-validation.

In this final installment, we will pull back the curtain to see how these models actually carve up the feature space. We will visualize decision boundaries—the geometric lines and shapes that separate one class from another. We will also learn how to calculate precision, recall, and f1-score when we have 3 or more classes using macro, weighted, and micro averaging.

### Learning Objectives
By the end of this notebook, you will be able to:
1.  Visualize decision boundaries in 2D using PCA.
2.  Compare the geometric logic of logistic regression (linear) vs. decision trees (axis-aligned) vs. KNN (non-linear).
3.  Interpret multiclass metrics (Macro vs. Weighted vs. Micro averaging).
4.  Understand feature importance in a multiclass context.

## Section 1: Setup and Data Loading (Wine Dataset)

We will continue with the wine dataset introduced in Part 4.

In [ ]:
from xgboost import XGBClassifier
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.decomposition import PCA

wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


print(f"Setup complete. Classes: {wine.target_names}")
print(f"Class Distribution: {np.bincount(y)}")
print(f"Features: {len(wine.feature_names)}")

## Section 2: Visualizing Decision Boundaries

It's hard to visualize 13 chemical features at once. To see the "decision boundaries," we use PCA (Principal Component Analysis) to squash the data into its two most important dimensions (PC1 and PC2).

PCA finds the directions in the data that capture the most variance. PC1 is the direction of maximum variance; PC2 is the direction of maximum variance orthogonal to PC1. Together, they give us the best 2D view of the data's structure.

In [ ]:
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train)

explained = pca.explained_variance_ratio_
print(f"PC1 explains {explained[0]*100:.1f}% of variance")
print(f"PC2 explains {explained[1]*100:.1f}% of variance")
print(f"Together: {explained.sum()*100:.1f}% of total variance")
print(f"\nThis means we're seeing {explained.sum()*100:.0f}% of the data's structure in 2D.")

Now let's train three different models on these 2D features and plot their "territories."

**Note on KNN:** K-Nearest Neighbors classifies each point by looking at its K closest neighbors in the training set. Unlike logistic regression (which learns a global rule), KNN makes local decisions based on nearby points.

In [ ]:
def plot_boundaries(clf, X, y, ax, title):
    h = .02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='RdYlBu', s=50)
    
    # Add test accuracy to the title
    acc = clf.score(X, y)
    ax.set_title(f"{title}\n(Train Acc = {acc:.3f})")
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

clfs = [
    ('Logistic Regression (Linear)', XGBClassifier(objective="multi:softprob", num_class=3, use_label_encoder=False, eval_metric="mlogloss", random_state=42)),
    ('Decision Tree (Axis-Aligned)', XGBClassifier(max_depth=3, objective='multi:softprob', num_class=3, eval_metric='mlogloss', random_state=42)),
    ('KNN (Non-linear)', XGBClassifier(n_estimators=100, objective='multi:softprob', num_class=3, eval_metric='mlogloss', random_state=42))
]

for i, (name, clf) in enumerate(clfs):
    clf.fit(X_train_2d, y_train)
    plot_boundaries(clf, X_train_2d, y_train, axes[i], name)

plt.tight_layout()
plt.show()

### Reading the Decision Boundaries

Each plot shows the model's decision regions (colored backgrounds) overlaid with the training data points (dots). The boundaries between colored regions are where the model changes its prediction.

- **Logistic Regression (Linear):** The boundaries are straight lines. This model assumes the three wine types can be separated by linear cuts through the feature space. It's simple and interpretable, but may miss complex patterns.

- **Decision Tree (Axis-Aligned):** The boundaries are composed of horizontal and vertical steps. This is because trees split on one variable at a time: "Is PC1 > 2.5?" → yes/no → next split. The result is a "staircase" pattern. It's interpretable but rigid.

- **KNN (Local/Non-linear):** The boundaries are wavy and irregular. KNN looks at the neighbors around every single point, allowing it to adapt to very local patterns. It's flexible but can overfit to noise.

**Which model would you trust?** Logistic regression's simplicity makes it robust to overfitting, but it may underfit if the true boundaries are curved. KNN's flexibility captures complex patterns but may not generalize well. The Decision Tree sits in between—interpretable but limited to axis-aligned splits. The training accuracies shown in each subplot give you a quantitative comparison.

**Important caveat:** These boundaries are in PCA-reduced 2D space, not the original 13-dimensional space. The PCA projection captures {explained.sum()*100:.0f}% of the variance, so we're seeing most but not all of the structure. The true decision boundaries in 13D are more complex than what we see here.

### Summary: Geometric Model Behaviors

| Model | Boundary Shape | Strengths | Weaknesses |
|---|---|---|---|
| **Logistic Regression** | Straight lines (hyperplanes) | Simple, interpretable, robust | Can't capture non-linear patterns |
| **Decision Tree** | Axis-aligned rectangles | Interpretable rules, handles mixed data | Rigid boundaries, prone to overfitting |
| **KNN** | Wavy, local boundaries | Captures any shape, no training needed | Sensitive to noise, slow at prediction |
| **Random Forest** | Complex, non-linear | Robust, captures interactions | Less interpretable, many hyperparameters |
| **Gradient Boosting** | Complex, non-linear | Very accurate, handles complex patterns | Easy to overfit, slow to train |

## Section 3: Multiclass Confusion Matrix

When we have 3 classes, our confusion matrix becomes a 3×3 grid. Let's train a Random Forest on all 13 features and see where it's mixing up the wine types.

In [ ]:
# Train a Random Forest on ALL 13 features
model = XGBClassifier(objective="multi:softprob", num_class=3, n_estimators=100, eval_metric="mlogloss", random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(wine.target_names)
ax.set_yticklabels(wine.target_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('3×3 Multiclass Confusion Matrix')

# Add labels with counts
for i in range(3):
    for j in range(3):
        label = f'{cm[i, j]}'
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, label, ha='center', va='center', fontsize=16, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Interpreting the Confusion Matrix

The diagonal (top-left to bottom-right) shows correct predictions. Off-diagonal cells show misclassifications.

If two classes are frequently confused, it means their chemical properties overlap significantly. This could indicate that those two cultivars share similar growing conditions, grape varieties, or winemaking techniques.

## Section 4: Averaging Strategies (Macro vs. Weighted vs. Micro)

With 3 classes, you have 3 different precision scores (one for each wine). How do you get a single number?

- **Macro averaging:** Calculate the metric for each class separately, then take the simple average. This treats every class as equally important, regardless of how many samples it has.
- **Weighted averaging:** Calculate the metric for each class, but weight by the number of samples. This reflects overall performance on the dataset population.
- **Micro averaging:** Compute the metric globally across all samples. For precision and recall, this is equivalent to accuracy when every sample has exactly one label.

### When Do They Differ?

Imagine a dataset with 90 samples of class A and 10 of class B. A model that gets all A right but misses all B would have:
- **Macro precision = 50%** (A: 100%, B: 0%, average = 50%)
- **Weighted precision = 90%** (90 samples × 100% + 10 samples × 0% = 90%)
- **Micro precision = 90%** (90 correct out of 100 total)

Which one matters depends on your goals. If catching class B is critical (e.g., a rare disease), use macro. If overall performance is what matters, use weighted.

In [ ]:
print("Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

### Reading the Report

Notice that the macro and weighted averages are very similar. This is because the wine dataset is roughly balanced (59/71/48 samples). When classes are balanced, macro and weighted averages converge.

Look at the per-class scores: does the model perform equally well on all three wine types? If one class has noticeably lower precision or recall, that cultivar's chemical profile overlaps with the others.

## Section 5: Feature Importance

Finally, let's look at which features are most important for distinguishing these three cultivars. For tree-based models like random forest, we can use `feature_importances_` to see which variables led to the most "purity" when splitting the data.

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
bars = plt.bar(range(X.shape[1]), importances[indices], color='darkgreen', edgecolor='black')
plt.xticks(range(X.shape[1]), [wine.feature_names[i] for i in indices], rotation=45, ha='right')
plt.title('What Drives Wine Classification? (Feature Importance)')
plt.ylabel('Relative Importance')

# Add value labels on top of each bar
for bar, idx in zip(bars, indices):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{importances[idx]:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

top_3 = [wine.feature_names[i] for i in indices[:3]]
top_3_vals = [importances[i] for i in indices[:3]]
print(f"Top 3 features: {', '.join(f'{f} ({v:.1%})' for f, v in zip(top_3, top_3_vals))}")

### Interpreting Feature Importance

The top features tell a chemical story about what differentiates these three cultivars:

- **Proline** is typically the strongest differentiator. Proline is an amino acid that varies significantly based on grape variety and soil composition. Different cultivars accumulate proline at different rates.
- **Color intensity** and **flavanoids** are also strong signals. These reflect the phenolic content of the wine, which is heavily influenced by grape variety and winemaking techniques.

Understanding these features allows you to explain *why* the model chose one cultivar over another—not just that it did. This interpretability is one of the key advantages of tree-based models over "black box" approaches.

## Conclusion

Congratulations! You have completed the comprehensive Introduction to Classification series.

**Series Recap:**
1.  **Part 1 (Foundations):** We learned the sigmoid function, maximum likelihood estimation, and how probability thresholds work.
2.  **Part 2 (Confusion Matrix):** We moved beyond accuracy to precision, recall, and f1-score—and saw how the decision threshold controls the trade-off.
3.  **Part 3 (Model Selection):** We compared multiple algorithms via cross-validation and explored how regularization shapes feature selection.
4.  **Part 4 (Multiclass):** We entered the multiclass world, comparing OvR vs. Softmax strategies and learning macro/weighted/micro averaging.
5.  Part 5 (Decision Boundaries): We visualized how different models carve up feature space, interpreted multiclass confusion matrices, and understood feature importance.

**Practical Takeaway:** When building a classification model, start with logistic regression as your baseline. Use cross-validation with multiple metrics (not just accuracy) to compare models. Always check the confusion matrix for class-specific weaknesses. And remember: the best model depends on your business costs, not just mathematical optimality.

In the next series, we will dive deeper into specific classification algorithms: logistic regression in detail (18_2), and ensemble methods for classification (18_5).